In [1]:
# ============================================================
# ResNet50V2 ASL (29 classes) — 3 DATASETS (train/val/test)
# + Multi-GPU (MirroredStrategy), Mixed Precision FP16, XLA JIT
# - Input: 3 thư mục Kaggle đã tách sẵn (mỗi thư mục gồm 29 class)
# - 224x224, chuẩn hóa [0,1], aug: rotation/flip/zoom/translation
# - Optimizer: Adam lr=0.001, Epochs=50, EarlyStopping
# - Loss: categorical_crossentropy
# - Metrics: Accuracy, Precision, Recall, Macro-F1
# - Outputs: figures/*.png, figures/classification_report.txt, artifacts/*.keras, label_map.json
# ============================================================
import os, json
from pathlib import Path

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# =========================
# GPU SETUP (memory growth + FP16 + multi-GPU)
# =========================
# Cho phép TF không chiếm hết VRAM ngay khi khởi động
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except Exception as e:
        print("Cannot set memory growth:", e)

# Mixed Precision (FP16) để tận dụng Tensor Cores (nếu có)
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")

# Multi-GPU (1 máy nhiều GPU). Nếu chỉ có 1 GPU, strategy vẫn hoạt động bình thường.
strategy = tf.distribute.MirroredStrategy()
NUM_GPUS = strategy.num_replicas_in_sync
print("GPUs in sync:", NUM_GPUS)

# -----------------------------
# CONFIG (ĐỔI 3 ĐƯỜNG DẪN NÀY CHO KHỚP DATASET CỦA BẠN)
# -----------------------------
TRAIN_DIR = "/kaggle/input/asl-alphabet/asl_split/train"   # <<< sửa
VAL_DIR   =  "/kaggle/input/asl-alphabet/asl_split/val"    # <<< sửa
TEST_DIR  = "/kaggle/input/asl-alphabet/asl_split/test"    # <<< sửa

IMG_SIZE   = (224, 224)
BASE_BATCH = 64 # batch per 1 GPU
BATCH_SIZE = BASE_BATCH * max(1, NUM_GPUS)  # scale batch theo số GPU
EPOCHS     = 50
SEED       = 123

os.makedirs("checkpoints", exist_ok=True)
os.makedirs("figures", exist_ok=True)
os.makedirs("artifacts", exist_ok=True)

# -----------------------------
# HÀM QUÉT FILES + LABELS
# -----------------------------
def list_image_files_with_labels(root_dir):
    root = Path(root_dir)
    if not root.exists():
        raise FileNotFoundError(f"Directory not found: {root_dir}")

    class_names = sorted([d.name for d in root.iterdir() if d.is_dir()])
    if len(class_names) == 0:
        raise ValueError(f"No class subfolders found under: {root_dir}")
    class_to_idx = {c: i for i, c in enumerate(class_names)}

    filepaths, labels = [], []
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    for c in class_names:
        for p in (root / c).rglob("*"):
            if p.suffix.lower() in exts:
                filepaths.append(str(p))
                labels.append(class_to_idx[c])
    return filepaths, labels, class_names

# -----------------------------
# ĐỌC 3 BỘ DỮ LIỆU
# -----------------------------
train_files, train_labels, train_classes = list_image_files_with_labels(TRAIN_DIR)
val_files,   val_labels,   val_classes   = list_image_files_with_labels(VAL_DIR)
test_files,  test_labels,  test_classes  = list_image_files_with_labels(TEST_DIR)

# Kiểm tra class set đồng nhất
if train_classes != val_classes or train_classes != test_classes:
    raise ValueError(
        "Class folders in TRAIN / VAL / TEST are not identical or not in the same order.\n"
        f"TRAIN: {train_classes}\nVAL  : {val_classes}\nTEST : {test_classes}\n"
        "Please ensure all three datasets contain the same 29 class subfolders with identical names."
    )

class_names = train_classes
num_classes = len(class_names)
print(f"[OK] Classes (29 expected): {num_classes}")
print(class_names)

print(f"Train images: {len(train_files)}")
print(f"Val   images: {len(val_files)}")
print(f"Test  images: {len(test_files)}")

assert num_classes == 29, "Expected 29 classes (26 letters + delete, nothing, space)."

2025-11-12 21:32:32.946544: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762983153.171321      48 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762983153.233382      48 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
GPUs in sync: 2


I0000 00:00:1762983169.637392      48 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1762983169.638052      48 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


[OK] Classes (29 expected): 29
['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space']
Train images: 55680
Val   images: 13920
Test  images: 17400


In [2]:
# -----------------------------
# TF.DATA PIPELINES (cache + prefetch + nondeterministic)
# -----------------------------
AUTOTUNE = tf.data.AUTOTUNE

def decode_img_onehot(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE, antialias=True)
    img = tf.cast(img, tf.float32) / 255.0   # scale [0,1]
    y = tf.one_hot(tf.cast(label, tf.int32), depth=num_classes)
    return img, y

augment = keras.Sequential([
    layers.RandomFlip("horizontal", seed=SEED),
    layers.RandomRotation(0.08, fill_mode="reflect", seed=SEED),      # ~±8%
    layers.RandomZoom(0.10, fill_mode="reflect", seed=SEED),          # ±10%
    layers.RandomTranslation(0.10, 0.10, fill_mode="reflect", seed=SEED),
], name="augment")

def make_ds(files, labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((files, labels))
    if training:
        ds = ds.shuffle(buffer_size=min(10000, len(files)), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(decode_img_onehot, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
        # Nếu RAM đủ lớn có thể cache train:
        # ds = ds.cache()
    else:
        # Val/Test nên cache (ổn định & nhanh)
        ds = ds.cache()

    # Cho phép non-deterministic để tăng throughput
    options = tf.data.Options()
    options.experimental_deterministic = False
    ds = ds.with_options(options)

    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_ds(train_files, train_labels, training=True)
val_ds   = make_ds(val_files,   val_labels,   training=False)
test_ds  = make_ds(test_files,  test_labels,  training=False)

# -----------------------------
# METRIC: Macro-F1 (từ confusion matrix tích lũy)
# -----------------------------
class MacroF1(keras.metrics.Metric):
    def __init__(self, num_classes, name="f1", **kwargs):
        super().__init__(name=name, **kwargs)
        self.num_classes = num_classes
        self.cm = self.add_weight(
            name="confusion_matrix",
            shape=(num_classes, num_classes),
            initializer="zeros",
            dtype=tf.float32
        )

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true_labels = tf.argmax(y_true, axis=1, output_type=tf.int32)
        y_pred_labels = tf.argmax(y_pred, axis=1, output_type=tf.int32)
        cm_batch = tf.math.confusion_matrix(
            y_true_labels, y_pred_labels, num_classes=self.num_classes, dtype=tf.float32
        )
        self.cm.assign_add(cm_batch)

    def result(self):
        tp = tf.linalg.diag_part(self.cm)
        fp = tf.reduce_sum(self.cm, axis=0) - tp
        fn = tf.reduce_sum(self.cm, axis=1) - tp

        precision = tf.math.divide_no_nan(tp, tp + fp)
        recall    = tf.math.divide_no_nan(tp, tp + fn)
        f1_per_c  = tf.math.divide_no_nan(2.0 * precision * recall, precision + recall)
        return tf.reduce_mean(f1_per_c)

    def reset_states(self):
        self.cm.assign(tf.zeros_like(self.cm))

# -----------------------------
# MODEL: ResNet50V2 + GAP + Dropout + Dense(29)
#  (build + compile trong strategy.scope để dùng đa-GPU)
# -----------------------------
with strategy.scope():
    base = keras.applications.ResNet50V2(
        include_top=False,
        weights="imagenet",
        input_shape=(*IMG_SIZE, 3),
        pooling=None
    )

    inputs = keras.Input(shape=(*IMG_SIZE, 3))
    x = base(inputs, training=True)  # end-to-end
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    # IMPORTANT: khi dùng mixed precision, ép output Dense sang float32 để ổn định loss/metrics
    outputs = layers.Dense(num_classes, activation="softmax", dtype="float32")(x)
    model = keras.Model(inputs, outputs, name="resnet50v2_asl")

    precision_metric = keras.metrics.Precision(name="precision")
    recall_metric    = keras.metrics.Recall(name="recall")
    f1_metric        = MacroF1(num_classes=num_classes, name="f1")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=3e-4),
        loss="categorical_crossentropy",
        metrics=["accuracy", precision_metric, recall_metric, f1_metric],
        # jit_compile=True  # XLA JIT tăng tốc thêm
    )

model.summary()

# -----------------------------
# CALLBACKS
# -----------------------------
ckpt_path = "checkpoints/resnet50v2_asl_best.h5"
callbacks = [
    keras.callbacks.TerminateOnNaN(),  # dừng sớm nếu loss NaN (đỡ “lây” cả epoch)
    keras.callbacks.ModelCheckpoint(
        ckpt_path, monitor="val_accuracy", mode="max",
        save_best_only=True, save_weights_only=False, verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=8, restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=4, verbose=1
    ),
]

# -----------------------------
# TRAIN
# -----------------------------
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)


94668760/94668760 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "resnet50v2_asl"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50v2 (Functional)         │ (None, 7, 7, 2048)     │    23,564,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cast_1 (Cast)                   │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 29)             │        59,421 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,624,221 (90.12 MB)

 Trainable params: 23,578,781 (89.95 MB)

 Non-trainable params: 45,440 (177.50 KB)

INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
Epoch 1/50
INFO:tensorflow:Collective all_reduce tensors: 1 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
INFO:tensorflow:Collective all_reduce tensors: 1 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
INFO:tensorflow:Collective all_reduce tensors: 174 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
INFO:tensorflow:Collective all_reduce tensors: 1 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
INFO:tensorflow:Collective all_reduce tensors: 1 all_reduces, num_d

I0000 00:00:1762983338.963468     112 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1762983338.988281     109 cuda_dnn.cc:529] Loaded cuDNN version 90300


119/435 ━━━━━━━━━━━━━━━━━━━━ 7:25 1s/step - accuracy: 0.2893 - f1: 0.0678 - loss: 2.7049 - precision: 0.9820 - recall: 0.1038Batch 119: Invalid loss, terminating training
120/435 ━━━━━━━━━━━━━━━━━━━━ 7:24 1s/step - accuracy: 0.2894 - f1: 0.0678 - loss: inf - precision: 0.9820 - recall: 0.1038   INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Collective all_reduce tensors: 1 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
INFO:tensorflow:Collective all_reduce tensors: 1 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
INFO:tensorflow:Collective all_reduce tensors: 1 all_reduces, num_devices = 2, group_size = 2, implement

435/435 ━━━━━━━━━━━━━━━━━━━━ 254s 470ms/step - accuracy: 0.2927 - f1: 0.0686 - loss: inf - precision: 0.9878 - recall: 0.1061 - val_accuracy: 0.9980 - val_f1: 0.3149 - val_loss: 0.0103 - val_precision: 0.9981 - val_recall: 0.9979 - learning_rate: 3.0000e-04
Restoring model weights from the end of the best epoch: 1.


In [ ]:
# -----------------------------
# EVALUATE (overall metrics)
# -----------------------------
test_metrics = model.evaluate(test_ds, verbose=1)
print("\n=== TEST METRICS ===")
for name, val in zip(model.metrics_names, test_metrics):
    print(f"{name}: {val:.6f}")

# -----------------------------
# PLOT TRAINING CURVES
# -----------------------------
import matplotlib.pyplot as plt

hist = history.history  # dict

def plot_curve(keys, title, fname):
    plt.figure(figsize=(7,5))
    for k in keys:
        if k in hist:
            plt.plot(hist[k], label=k)
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel("Value")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(f"figures/{fname}", dpi=150, bbox_inches="tight")
    plt.close()

plot_curve(["loss", "val_loss"], "Loss", "loss.png")
plot_curve(["accuracy", "val_accuracy"], "Accuracy", "accuracy.png")
plot_curve(["precision", "val_precision"], "Precision", "precision.png")
plot_curve(["recall", "val_recall"], "Recall", "recall.png")
plot_curve(["f1", "val_f1"], "Macro-F1", "f1.png")
print("Saved training curves to figures/*.png")

# -----------------------------
# CONFUSION MATRIX & REPORT (test set)
# -----------------------------
from sklearn.metrics import confusion_matrix, classification_report

y_true = []
y_pred = []

for batch_x, batch_y in test_ds:
    preds = model.predict(batch_x, verbose=0)
    y_pred.append(np.argmax(preds, axis=1))
    y_true.append(np.argmax(batch_y.numpy(), axis=1))

y_true = np.concatenate(y_true)
y_pred = np.concatenate(y_pred)

cm = confusion_matrix(y_true, y_pred, labels=np.arange(num_classes))
cm_norm = cm.astype(np.float32) / np.maximum(cm.sum(axis=1, keepdims=True), 1)

plt.figure(figsize=(10,8))
plt.imshow(cm_norm, interpolation="nearest")
plt.title("Confusion Matrix (normalized by row)")
plt.colorbar()
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.savefig("figures/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.close()

report = classification_report(y_true, y_pred, target_names=class_names, digits=4)
print("\n=== Classification Report (per class) ===")
print(report)
with open("figures/classification_report.txt", "w", encoding="utf-8") as f:
    f.write(report)

print("Saved confusion matrix and report to figures/")

# -----------------------------
# SAVE ARTIFACTS
# -----------------------------
model.save("artifacts/resnet50v2_asl.keras")
with open("artifacts/label_map.json", "w", encoding="utf-8") as f:
    json.dump({i: c for i, c in enumerate(class_names)}, f, ensure_ascii=False, indent=2)

print("\nSaved model -> artifacts/resnet50v2_asl.keras")
print("Saved label map -> artifacts/label_map.json")

INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).


INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).


INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).


INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).


 32/136 ━━━━━━━━━━━━━━━━━━━━ 23s 230ms/step - accuracy: 0.9995 - f1: 0.1103 - loss: 0.0034 - precision: 0.9995 - recall: 0.9994